In [78]:
import pandas as pd
import pandas as pd
import json
import os

with open("../config.json") as json_file:
    _LOCAL_CONFIG = json.load(json_file)

milk10k_folder_path = _LOCAL_CONFIG["milk10k_folder_path"]
dataset_output_path = _LOCAL_CONFIG["output_folder_path"]

In [79]:
df = pd.read_csv(os.path.join(milk10k_folder_path, 'MILK10k_Training_Metadata.csv'))
df_ground_truth = pd.read_csv(os.path.join(milk10k_folder_path, 'MILK10k_Training_GroundTruth.csv'))

In [80]:
df.head(2)

,lesion_id,image_type,isic_id,attribution,copyright_license,image_manipulation,age_approx,sex,skin_tone_class,site,MONET_ulceration_crust,MONET_hair,MONET_vasculature_vessels,MONET_erythema,MONET_pigmented,MONET_gel_water_drop_fluid_dermoscopy_liquid,MONET_skin_markings_pen_ink_purple_pen
0,IL_0000652,clinical: close-up,ISIC_8149219,MILK study team,CC-BY-NC,altered,70.0,male,1,head_neck_face,0.166749,0.163601,0.002284,0.124315,0.719495,0.220399,0.237601
1,IL_0000652,dermoscopic,ISIC_4671410,MILK study team,CC-BY-NC,instrument only,70.0,male,1,head_neck_face,0.659859,0.156478,0.016397,0.032357,0.847014,0.138121,0.148776


In [81]:
df_ground_truth.head(2)

,lesion_id,AKIEC,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC
0,IL_0000652,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,IL_0003176,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [82]:
df_ground_truth.columns

Index(['lesion_id', 'AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH',
       'MEL', 'NV', 'SCCKA', 'VASC'],
      dtype='object')

In [83]:
colunas_diag = df_ground_truth.columns.drop('lesion_id')
df_diagnostic = df_ground_truth[['lesion_id']].copy()
df_diagnostic['diagnostic'] = df_ground_truth[colunas_diag].idxmax(axis=1)

In [84]:
df_diagnostic.head(2)

,lesion_id,diagnostic
0,IL_0000652,BCC
1,IL_0003176,BCC


In [85]:
df = pd.merge(df, df_diagnostic, on='lesion_id')
df.head(2)

,lesion_id,image_type,isic_id,attribution,copyright_license,image_manipulation,age_approx,sex,skin_tone_class,site,MONET_ulceration_crust,MONET_hair,MONET_vasculature_vessels,MONET_erythema,MONET_pigmented,MONET_gel_water_drop_fluid_dermoscopy_liquid,MONET_skin_markings_pen_ink_purple_pen,diagnostic
0,IL_0000652,clinical: close-up,ISIC_8149219,MILK study team,CC-BY-NC,altered,70.0,male,1,head_neck_face,0.166749,0.163601,0.002284,0.124315,0.719495,0.220399,0.237601,BCC
1,IL_0000652,dermoscopic,ISIC_4671410,MILK study team,CC-BY-NC,instrument only,70.0,male,1,head_neck_face,0.659859,0.156478,0.016397,0.032357,0.847014,0.138121,0.148776,BCC


In [86]:
df.columns

Index(['lesion_id', 'image_type', 'isic_id', 'attribution',
       'copyright_license', 'image_manipulation', 'age_approx', 'sex',
       'skin_tone_class', 'site', 'MONET_ulceration_crust', 'MONET_hair',
       'MONET_vasculature_vessels', 'MONET_erythema', 'MONET_pigmented',
       'MONET_gel_water_drop_fluid_dermoscopy_liquid',
       'MONET_skin_markings_pen_ink_purple_pen', 'diagnostic'],
      dtype='object')

In [87]:
df.drop(columns=['attribution',
       'copyright_license', 'image_manipulation',
       'skin_tone_class', 'MONET_ulceration_crust', 'MONET_hair',
       'MONET_vasculature_vessels', 'MONET_erythema', 'MONET_pigmented',
       'MONET_gel_water_drop_fluid_dermoscopy_liquid',
       'MONET_skin_markings_pen_ink_purple_pen'
], inplace=True)
df.dropna(inplace=True)

In [88]:
body_map = {
    'head_neck_face' : 'CABECA',
    'lower_extremity': 'DORSO',
    'upper_extremity' : 'PEITORAL',
    'trunk' : 'DORSO',
    'foot' : 'PE', 
    'genital' : 'GENITARIA',
    'hand' : 'MAO'
}
df['site'] = df['site'].map(body_map)

In [89]:
df['diagnostic'].unique()

array(['BCC', 'SCCKA', 'AKIEC', 'NV', 'BKL', 'MEL', 'BEN_OTH', 'DF',
       'MAL_OTH', 'VASC', 'INF'], dtype=object)

In [90]:
diag_map = {
    'MEL' : 'C43',
    'NV' : 'D22',
    'BCC' : 'C80',
    'SCCKA' : 'C44',
    'AKIEC' : 'L57',
    'BKL' : 'L82',
}

df['clinicalMacroCID'] = df['diagnostic'].map(diag_map)
df.head(2)

,lesion_id,image_type,isic_id,age_approx,sex,site,diagnostic,clinicalMacroCID
0,IL_0000652,clinical: close-up,ISIC_8149219,70.0,male,CABECA,BCC,C80
1,IL_0000652,dermoscopic,ISIC_4671410,70.0,male,CABECA,BCC,C80


In [91]:
df['sex'].unique()

array(['male', 'female'], dtype=object)

In [92]:
gender_map = {
    'male' : 'M',
    'female' : 'F'
}

df['sex'] = df['sex'].map(gender_map)

In [93]:
df['image_type'].unique()

array(['clinical: close-up', 'dermoscopic'], dtype=object)

In [94]:
img_type_map = {
    'clinical: close-up' : 'CLINICAL',
    'dermoscopic' : 'DERMATOSCOPE'
}

df['image_type'] = df['image_type'].map(img_type_map)

In [95]:
df.columns

Index(['lesion_id', 'image_type', 'isic_id', 'age_approx', 'sex', 'site',
       'diagnostic', 'clinicalMacroCID'],
      dtype='object')

In [96]:
df['img-id'] = df.apply(lambda x: os.path.join(milk10k_folder_path, 'MILK10k_Training_Input', x['lesion_id'], x['isic_id'] + '.jpg'), axis=1)

In [97]:
df['img-id'].iloc[300]

'/home/epmagesk/datasets/MILK10k/MILK10k_Training_Input/IL_0285359/ISIC_1175016.jpg'

In [98]:
df.drop(columns=['isic_id'], inplace=True)
df.rename(columns={'image_type' : 'img-src', 
                   'lesion_id' : 'lesion-id', 
                   'sex' : 'gender', 
                   'age_approx' : 'age', 
                   'site' : 'macroBodyRegion',
                   'diagnostic' : 'clinicalDiagnostic'}, inplace=True)
df.head(2)

,lesion-id,img-src,age,gender,macroBodyRegion,clinicalDiagnostic,clinicalMacroCID,img-id
0,IL_0000652,CLINICAL,70.0,M,CABECA,BCC,C80,/home/epmagesk/datasets/MILK10k/MILK10k_Traini...
1,IL_0000652,DERMATOSCOPE,70.0,M,CABECA,BCC,C80,/home/epmagesk/datasets/MILK10k/MILK10k_Traini...


In [99]:
# df['usePesticide'] = 'I'
# df['familySkinCancerHistory'] = 'I'
# df['familyCancerHistory'] = 'I'
# df['hasItched'] = 'I'
# df['hasGrown'] = 'I'
# df['hasHurt'] = 'I'
# df['hasChanged'] = 'I'
# df['hasBled'] = 'I'
# df['hasElevation'] = 'I'
df['histoDiagnostic'] = 'EMPTY'
df['histoMacroCID'] = 'EMPTY'

df.head(2)

,lesion-id,img-src,age,gender,macroBodyRegion,clinicalDiagnostic,clinicalMacroCID,img-id,histoDiagnostic,histoMacroCID
0,IL_0000652,CLINICAL,70.0,M,CABECA,BCC,C80,/home/epmagesk/datasets/MILK10k/MILK10k_Traini...,EMPTY,EMPTY
1,IL_0000652,DERMATOSCOPE,70.0,M,CABECA,BCC,C80,/home/epmagesk/datasets/MILK10k/MILK10k_Traini...,EMPTY,EMPTY


In [100]:
df.dropna(inplace=True)
df['clinicalDiagnostic'].value_counts()


clinicalDiagnostic
BCC      5036
NV       1424
BKL      1082
SCCKA     946
MEL       888
AKIEC     606
Name: count, dtype: int64

In [101]:
df.to_csv(os.path.join(dataset_output_path, 'milk10k_cli_derm_metadata.csv'), index=False)